In [3]:
pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 1.1 MB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 1.5 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 1.8 MB/s eta 0:00:01
   ------------------------------------- -- 1.8/2.0 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 1.8 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

# ------------------------------------
# PARAMETERS
# ------------------------------------
YEARS = 10
DOCTORS_COUNT = 213
PATIENTS_COUNT = 35456
DEPARTMENTS = [
    "Cardiology", "Neurology", "Orthopedics", "Oncology", "Dermatology",
    "Pediatrics", "ENT", "Ophthalmology", "Urology", "Gastroenterology",
    "General Medicine", "Nephrology", "Gynecology", "Surgery"
]

# ------------------------------------
# DIM DEPARTMENT
# ------------------------------------
dim_department = pd.DataFrame({
    "department_id": range(1, len(DEPARTMENTS)+1),
    "department_name": DEPARTMENTS,
    "floor_no": [random.randint(1,5) for _ in DEPARTMENTS],
    "head_doctor_id": [random.randint(1, DOCTORS_COUNT) for _ in DEPARTMENTS]
})

# ------------------------------------
# DIM DOCTOR
# ------------------------------------
specializations = DEPARTMENTS
doctors = []
for i in range(1, DOCTORS_COUNT+1):
    dept = random.choice(DEPARTMENTS)
    doctors.append([
        i,
        fake.name(),
        random.choice(["Male","Female"]),
        random.randint(28,65),
        dept,
        list(dim_department[dim_department['department_name']==dept]['department_id'])[0],
        fake.date_between(start_date='-15y', end_date="-1y"),
        random.randint(1,35),
        random.choice(["Active","Active","Active","Resigned"])      
    ])

dim_doctor = pd.DataFrame(doctors, columns=[
    "doctor_id","doctor_name","gender","age","specialization",
    "department_id","join_date","experience_years","status"
])

# ------------------------------------
# DIM PATIENT
# ------------------------------------
patients = []
for i in range(1, PATIENTS_COUNT+1):
    patients.append([
        i, fake.name(),
        random.choice(["Male","Female"]),
        random.randint(1,90),
        random.choice(["A+","A-","B+","B-","O+","O-","AB+","AB-"]),
        fake.city(),
        fake.date_between(start_date="-10y", end_date="today"),
        random.choice(["Diabetes","Hypertension","None","None","None","Asthma"]),
        random.choice(["Private","Government","Self Pay"])
    ])

dim_patient = pd.DataFrame(patients, columns=[
    "patient_id","patient_name","gender","age","blood_group",
    "city","registration_date","chronic_condition","insurance_type"
])

# ------------------------------------
# DIM SLOTS (Generate for last 10 years)
# ------------------------------------
slot_rows = []
slot_id = 1
start_date = datetime.now() - timedelta(days=YEARS*365)

for day in pd.date_range(start=start_date, end=datetime.now()):
    for doctor in range(1, DOCTORS_COUNT+1):
        daily_slots = random.randint(8, 15)
        for _ in range(daily_slots):
            start_t = fake.time_object()
            duration = random.choice([15,20,30])
            end_t = (datetime.combine(datetime.today(), start_t) + timedelta(minutes=duration)).time()

            slot_rows.append([
                slot_id,
                doctor,
                day.date(),
                start_t,
                end_t,
                random.choices(["Consultation","Follow-up","Emergency"], weights=[70,15,15])[0]
            ])
            slot_id += 1

dim_slots = pd.DataFrame(slot_rows, columns=[
    "slot_id","doctor_id","slot_date","slot_start_time","slot_end_time","slot_type"
])

# ------------------------------------
# FACT APPOINTMENTS
# ------------------------------------
appointments = []
appointment_id = 1

for index, slot in dim_slots.iterrows():
    if random.random() < 0.65:   # 65% slot utilisation
        appointments.append([
            appointment_id,
            random.randint(1, PATIENTS_COUNT),
            slot.doctor_id,
            slot.slot_id,
            slot.slot_date,
            random.choice([1,0]),
            random.choices(["Completed","Cancelled","No Show"], weights=[85,8,7])[0],
            random.randint(10,30),
            random.randint(300,1500)
        ])
        appointment_id += 1

fact_appointments = pd.DataFrame(appointments, columns=[
    "appointment_id","patient_id","doctor_id","slot_id","appointment_date",
    "is_first_visit","appointment_status","consultation_duration","revenue"
])

# ------------------------------------
# FACT OPERATIONS
# ------------------------------------
operations = []
operation_id = 1
operation_types = ["Bypass Surgery","Tumor Removal","Fracture Repair","Appendix","Cataract","Kidney Stone","Cesarean"]

for _, app in fact_appointments.iterrows():
    if random.random() < 0.10:   # 10% of appointments → operations
        dept_id = dim_doctor.loc[dim_doctor['doctor_id']==app.doctor_id, 'department_id'].iloc[0]
        operations.append([
            operation_id,
            app.patient_id,
            app.doctor_id,
            dept_id,
            app.appointment_date,
            random.choice(operation_types),
            random.randint(30,240),
            random.choice(["Success","Complication","ICU Required"]),
            random.randint(5000,120000)
        ])
        operation_id += 1

fact_operations = pd.DataFrame(operations, columns=[
    "operation_id","patient_id","doctor_id","department_id","operation_date",
    "operation_type","duration_minutes","outcome","cost"
])

# ------------------------------------
# FACT ATTENDANCE
# ------------------------------------
attendance_rows = []
att_id = 1

for day in pd.date_range(start=start_date, end=datetime.now()):
    for doctor in range(1, DOCTORS_COUNT+1):
        is_present = random.choices([1,0], weights=[97,3])[0]
        if is_present:
            in_t = fake.time_object()
            out_t = fake.time_object()
        else:
            in_t, out_t = None, None

        attendance_rows.append([
            att_id, doctor, day.date(), in_t, out_t, is_present
        ])
        att_id += 1

fact_attendance = pd.DataFrame(attendance_rows, columns=[
    "attendance_id","doctor_id","attendance_date","in_time","out_time","is_present"
])

# ------------------------------------
# Display basic info
# ------------------------------------
print("Doctors:", dim_doctor.shape)
print("Patients:", dim_patient.shape)
print("Departments:", dim_department.shape)
print("Slots:", dim_slots.shape)
print("Appointments:", fact_appointments.shape)
print("Operations:", fact_operations.shape)
print("Attendance:", fact_attendance.shape)


Doctors: (213, 9)
Patients: (35456, 9)
Departments: (14, 4)
Slots: (8942078, 6)
Appointments: (5812673, 9)
Operations: (582359, 9)
Attendance: (777663, 6)


In [5]:
# ------------------------------------
# EXPORT TO CSV FILES
# ------------------------------------

dim_doctor.to_csv("dim_doctor.csv", index=False)
dim_patient.to_csv("dim_patient.csv", index=False)
dim_department.to_csv("dim_department.csv", index=False)
dim_slots.to_csv("dim_slots.csv", index=False)

fact_appointments.to_csv("fact_appointments.csv", index=False)
fact_operations.to_csv("fact_operations.csv", index=False)
fact_attendance.to_csv("fact_attendance.csv", index=False)

print("All CSV files have been created successfully!")


All CSV files have been created successfully!
